In [ ]:
!pip install -q requests torch bitsandbytes transformers sentencepiece accelerate gradio openai

In [ ]:
import os
import requests
from huggingface_hub import login
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
from IPython.display import Markdown, display, update_display
import gradio as gr
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
import torch

In [ ]:
AUDIO_MODEL = "whisper-1"
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"
audio_filename="Creador de Godot ｜ Juan Linietsky ｜ Entrevistas Press Over.mp3"
hf_token = os.getenv('HF_TOKEN')
#login(hf_token, add_to_git_credential=True)
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
openai = OpenAI()


In [ ]:
system_message = "Eres un asistente que produce resumenes de entrevistas a partir de transcripciones,con resumen, puntos clave de preguntas, conclusiones, en formato Markdown"

def message_user_pront(transcription):
    user_prompt = f"A continuación se muestra un extracto de la entrevista a Juan Liniesky uno de los fundadores de Godot. Por favor, escriba las actas en formato Markdown, incluyendo un resumen de la entrevista; las preguntas y respuestas; las conclusiones; y las acciones a tomar con los responsables.\n{transcription}"
    return user_prompt


def message_gpt(transcription):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": message_user_pront(transcription)}
    ]
    return messages

def quant():
    quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4")

    return quant_config

In [ ]:
def create_interview(audio_filename):
    audio_file = open(audio_filename, "rb")
    transcription = openai.audio.transcriptions.create(model=AUDIO_MODEL, file=audio_file, response_format="text")
    print(transcription)
    tokenizer = AutoTokenizer.from_pretrained(LLAMA)
    tokenizer.pad_token = tokenizer.eos_token
    inputs = tokenizer.apply_chat_template(message_gpt(transcription), return_tensors="pt")
    streamer = TextStreamer(tokenizer)
    model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="cuda", quantization_config=quant())
    outputs = model.generate(inputs['input_ids'], max_new_tokens=2000, streamer=streamer)
    response = tokenizer.decode(outputs[0],skip_special_tokens=True)
    audio_file.close()
    return response
    
    

In [ ]:
def create_interview_model(audio_filename,model):
    if model=="GPT":
        result = create_interview(audio_filename)
        return result

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("""
    # Resumen Entrevista!
    Sube el audio de la entrevista.""")
    with gr.Row():
        audio = gr.Audio(label="Sube tu audio", type="filepath")
        out = gr.Textbox(label="Resumen")
    with gr.Row():
        model= gr.Dropdown(["GPT"],label="Seleciona el modelo",value="GPT")
        convert = gr.Button("Realizar resumen")
            
        convert.click(create_interview_model, inputs=[audio,model], outputs=[out])

demo.launch()